# 01 — Train and evaluate

This notebook is an execution guide. All environment, training, evaluation, statistics, plotting, and synchronization logic remains in the three repository Python files.

## SECTION 0 — User settings

In [ ]:
REPO_URL = "https://github.com/djdhillxn/carretera"
REPO_BRANCH = "main"
REPO_DIR = "/content/carla-highway-rl"
DRIVE_ROOT = "/content/drive/MyDrive/CARLA_Highway_RL"
CARLA_SERVER_MODE = "external"
CARLA_HOST = "127.0.0.1"
CARLA_PORT = 2000
CARLA_TM_PORT = 8000
CARLA_ROOT = ""
RUN_NAME = "ppo_seed_0"
SEED = 0
RUN_TINY_SANITY = False

## SECTION 1 — Common initialization

CARLA's Python client and simulator server are separate. The same repository code supports: (1) a packaged server in this Linux runtime using managed mode, (2) an external GPU machine, or (3) a Colab frontend connected to a local/remote runtime with CARLA installed. The packaged server is never downloaded or built here.

Dependencies are installed before this notebook imports NumPy, CARLA, Gymnasium, or SB3. The requirements accept the runtime's existing NumPy 2 release, so a normal Colab run does not downgrade NumPy or require a restart. The cell checks for a real loaded-versus-installed version mismatch and validates all binary imports in a fresh subprocess before continuing.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

repo = Path(REPO_DIR)
if not repo.exists():
    if REPO_URL == "REPLACE_WITH_GITHUB_URL":
        raise RuntimeError("Set REPO_URL before cloning in a hosted runtime.")
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(repo)], check=True)
else:
    status = subprocess.run(["git", "status", "--porcelain"], cwd=repo, check=True, capture_output=True, text=True)
    print(status.stdout or "Working tree clean.")
    if not status.stdout.strip():
        subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=repo, check=True)
        subprocess.run(["git", "merge", "--ff-only", "FETCH_HEAD"], cwd=repo, check=True)
    else:
        print("Local edits preserved; automatic update skipped.")

os.chdir(repo)
os.environ.update({
    "CARLA_HOST": CARLA_HOST,
    "CARLA_PORT": str(CARLA_PORT),
    "CARLA_TM_PORT": str(CARLA_TM_PORT),
    "CARLA_ROOT": CARLA_ROOT,
    "CARLA_SERVER_MODE": CARLA_SERVER_MODE,
    "HIGHWAY_RL_ARTIFACT_ROOT": str(repo / "artifacts"),
    "HIGHWAY_RL_DRIVE_ROOT": DRIVE_ROOT,
})
print("Repository:", repo)
print("Artifacts:", os.environ["HIGHWAY_RL_ARTIFACT_ROOT"])
print("Drive:", DRIVE_ROOT)

numpy_loaded = sys.modules.get("numpy")
numpy_loaded_version = getattr(numpy_loaded, "__version__", None)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

import importlib.metadata
numpy_installed_version = importlib.metadata.version("numpy")
if numpy_loaded_version and numpy_loaded_version != numpy_installed_version:
    raise RuntimeError(f"NumPy changed from loaded {numpy_loaded_version} to installed {numpy_installed_version}. Restart this runtime once, then rerun Sections 0–1.")
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
subprocess.run([sys.executable, "-c", "import carla, cv2, gymnasium, matplotlib, numpy, pandas, stable_baselines3, torch, yaml; print('Dependency imports OK; NumPy', numpy.__version__)"], check=True)
for package in ["carla", "numpy", "gymnasium", "stable-baselines3", "pandas", "matplotlib", "PyYAML", "torch", "opencv-python-headless"]:
    try:
        print(package, importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError:
        print(package, "not installed")

subprocess.run([sys.executable, "run.py", "sync", "--config", "config.yaml", "--from-drive", "--local-root", "artifacts", "--drive-root", DRIVE_ROOT], check=True)
subprocess.run([sys.executable, "run.py", "validate-config", "--config", "config.yaml"], check=True)

## SECTION 2 — Runtime and CARLA doctor

The doctor checks versions, map availability, Traffic Manager, synchronous settings, disk/GPU context, and writes deterministic highway candidates.

In [ ]:
import shutil
print("Disk:", shutil.disk_usage(REPO_DIR))
subprocess.run(["nvidia-smi"], check=False)
if CARLA_SERVER_MODE == "managed":
    subprocess.run([sys.executable, "run.py", "server", "start", "--config", "config.yaml"], check=True)
subprocess.run([sys.executable, "run.py", "doctor", "--config", "config.yaml"], check=True)
candidate_path = Path("artifacts/logs/runtime/highway_candidates.json")
print(candidate_path.read_text()[:5000])

## SECTION 3 — Environment and baseline smoke tests

The second command is server-backed and stops immediately on either environment-checker failure.

In [ ]:
subprocess.run([sys.executable, "run.py", "offline-self-test", "--config", "config.yaml"], check=True)
subprocess.run([sys.executable, "run.py", "smoke", "--config", "config.yaml"], check=True)
print(Path("artifacts/logs/runtime/smoke_test.json").read_text())

## SECTION 4 — Optional tiny PPO sanity run

This separate 2,000-decision run checks plumbing only; it is not the final model and may be rerun under a new name.

In [ ]:
if RUN_TINY_SANITY:
    tiny_name = f"tiny_sanity_seed_{SEED}"
    tiny_model = Path("artifacts/models") / tiny_name / "final_model.zip"
    if tiny_model.exists():
        print("Tiny sanity model already exists; not overwriting:", tiny_model)
    else:
        subprocess.run([sys.executable, "run.py", "train", "--config", "config.yaml", "--run-name", tiny_name, "--seed", str(SEED), "--total-timesteps", "2000"], check=True)
else:
    print("Tiny sanity run disabled. Set RUN_TINY_SANITY=True to execute it.")

## SECTION 5 — Primary 25K training stage

In [ ]:
stage_model = Path("artifacts/models") / RUN_NAME / "final_model.zip"
if stage_model.exists():
    print("Primary-stage model exists; refusing silent overwrite:", stage_model)
else:
    subprocess.run([sys.executable, "run.py", "train", "--config", "config.yaml", "--run-name", RUN_NAME, "--seed", str(SEED), "--total-timesteps", "25000"], check=True)
subprocess.run([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive", "--local-root", "artifacts", "--drive-root", DRIVE_ROOT], check=True)
from IPython.display import display, Image
for image in Path("artifacts/plots").glob("training_*.png"):
    display(Image(filename=str(image)))

## SECTION 6 — Resume to 50K

Restore Drive first, locate the latest checkpoint, and add 25K decisions without resetting the SB3 timestep counter.

In [ ]:
subprocess.run([sys.executable, "run.py", "sync", "--config", "config.yaml", "--from-drive", "--local-root", "artifacts", "--drive-root", DRIVE_ROOT], check=True)
checkpoints = sorted((Path("artifacts/models") / RUN_NAME / "checkpoints").glob("*.zip"))
if not checkpoints:
    raise RuntimeError("No valid checkpoint found for the resume stage.")
latest = checkpoints[-1]
print("Resuming:", latest)
subprocess.run([sys.executable, "run.py", "train", "--config", "config.yaml", "--run-name", RUN_NAME, "--seed", str(SEED), "--resume", str(latest), "--additional-timesteps", "25000"], check=True)
subprocess.run([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive", "--local-root", "artifacts", "--drive-root", DRIVE_ROOT], check=True)

## SECTION 7 — Freeze evaluation manifest

The command creates 90 paired conditions and refuses an existing path unless `--force` is deliberately supplied.

In [ ]:
manifest = Path("artifacts/manifests/evaluation_manifest.json")
if not manifest.exists():
    subprocess.run([sys.executable, "run.py", "make-eval-manifest", "--config", "config.yaml"], check=True)
import json
manifest_data = json.loads(manifest.read_text())
print("Rows:", manifest_data["scenario_count"], "hash:", manifest_data["manifest_hash"])
subprocess.run([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive", "--local-root", "artifacts", "--drive-root", DRIVE_ROOT], check=True)

## SECTION 8 — Quick evaluation

Runs the three held-out quick seeds for all four policies so failures can be inspected before committing to the full matrix.

In [ ]:
model = Path("artifacts/models") / RUN_NAME / "final_model.zip"
command = [sys.executable, "run.py", "evaluate", "--config", "config.yaml", "--manifest", str(manifest), "--model", str(model), "--policies", "ppo", "random", "keep_lane", "rule_based", "--quick"]
if Path("artifacts/evaluations/episode_results.csv").exists():
    command.append("--resume-existing")
subprocess.run(command, check=True)
import pandas as pd
quick = pd.read_csv("artifacts/evaluations/episode_results.csv")
display(quick.groupby("policy")[["success", "collision", "route_completion", "mean_speed_kmh"]].mean())

## SECTION 9 — Full evaluation

The matrix is evaluated in restartable 15-condition slices. Completed policy/condition pairs are skipped and partial CSVs are synchronized after each slice.

In [ ]:
for start in range(0, 90, 15):
    command = [sys.executable, "run.py", "evaluate", "--config", "config.yaml", "--manifest", str(manifest), "--model", str(model), "--policies", "ppo", "random", "keep_lane", "rule_based", "--start-index", str(start), "--end-index", str(min(start + 15, 90)), "--resume-existing"]
    subprocess.run(command, check=True)
    subprocess.run([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive", "--local-root", "artifacts", "--drive-root", DRIVE_ROOT], check=True)
episodes = pd.read_csv("artifacts/evaluations/episode_results.csv")
print("Completed rows:", len(episodes), "expected: 360")

## SECTION 10 — Analysis

In [ ]:
subprocess.run([sys.executable, "run.py", "analyze", "--config", "config.yaml", "--episodes", "artifacts/evaluations/episode_results.csv"], check=True)
summary = pd.read_csv("artifacts/evaluations/summary_results.csv")
display(summary[summary["group_type"] == "overall"])
for image in sorted(Path("artifacts/plots").glob("*.png")):
    display(Image(filename=str(image)))
subprocess.run([sys.executable, "run.py", "report-data", "--config", "config.yaml"], check=True)
subprocess.run([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive", "--local-root", "artifacts", "--drive-root", DRIVE_ROOT], check=True)

## SECTION 11 — Shutdown

In [ ]:
subprocess.run([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive", "--local-root", "artifacts", "--drive-root", DRIVE_ROOT], check=True)
if CARLA_SERVER_MODE == "managed":
    subprocess.run([sys.executable, "run.py", "server", "stop", "--config", "config.yaml"], check=True)
else:
    print("External CARLA server left untouched.")
print("Local artifacts:", Path("artifacts").resolve())
print("Drive artifacts:", DRIVE_ROOT)